# Model Evaluation & Interpretation
**Digital Banking Fraud Detection Project**

This notebook evaluates the performance of our 4 trained models on the test set, plots performance comparisons, generates feature importance visualizations, and saves the best performing model for Streamlit deployment.

In [ ]:
import os
import pickle
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, 
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)

### Load Test Data and Models

In [ ]:
X_test = pd.read_csv('../Models/X_test.csv')
y_test = pd.read_csv('../Models/y_test.csv').values.ravel()

models = {}
for model_name in ['logistic_regression', 'decision_tree', 'random_forest', 'xgboost']:
    with open(f'../Models/{model_name}.pkl', 'rb') as f:
        models[model_name] = pickle.load(f)
print("Loaded test dataset and all models successfully.")

### Compute Performance Metrics

In [ ]:
metrics = []

for name, model in models.items():
    y_pred = model.predict(X_test)
    if hasattr(model, 'predict_proba'):
        y_prob = model.predict_proba(X_test)[:, 1]
    else:
        y_prob = model.decision_function(X_test)
        
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    
    metrics.append({
        'Model': name.replace('_', ' ').title(),
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1,
        'ROC-AUC': auc
    })

metrics_df = pd.DataFrame(metrics).sort_values(by='ROC-AUC', ascending=False)
metrics_df

### Confusion Matrices Plot

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for idx, (name, model) in enumerate(models.items()):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx], cbar=False,
                xticklabels=['Legitimate', 'Fraudulent'], yticklabels=['Legitimate', 'Fraudulent'])
    axes[idx].set_title(f"{name.replace('_', ' ').title()} Confusion Matrix")
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('True')

plt.tight_layout()
plt.show()

### ROC Curves Plot Overlay

In [ ]:
plt.figure(figsize=(10, 8))

for name, model in models.items():
    if hasattr(model, 'predict_proba'):
        y_prob = model.predict_proba(X_test)[:, 1]
    else:
        y_prob = model.decision_function(X_test)
        
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    plt.plot(fpr, tpr, label=f"{name.replace('_', ' ').title()} (AUC = {auc:.4f})")

plt.plot([0, 1], [0, 1], 'k--', label='Random Guess')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves Comparison')
plt.legend(loc="lower right")
plt.grid(True)
plt.show()

### Detailed Classification Report for Best Model

In [ ]:
# Identify best model based on ROC-AUC
best_model_name = metrics_df.iloc[0]['Model'].replace(' ', '_').lower()
best_model = models[best_model_name]

y_pred = best_model.predict(X_test)
print(f"Best Model: {best_model_name.replace('_', ' ').title()}\n")
print(classification_report(y_test, y_pred, target_names=['Legitimate', 'Fraudulent']))

### Feature Importance

In [ ]:
features = list(X_test.columns)

plt.figure(figsize=(14, 12))

# Random Forest Feature Importance
rf_importances = models['random_forest'].feature_importances_
rf_indices = np.argsort(rf_importances)[::-1][:15]

plt.subplot(2, 1, 1)
sns.barplot(x=rf_importances[rf_indices], y=[features[i] for i in rf_indices], palette='viridis')
plt.title('Random Forest Feature Importance (Top 15)')
plt.xlabel('Relative Importance')

# XGBoost Feature Importance
xgb_importances = models['xgboost'].feature_importances_
xgb_indices = np.argsort(xgb_importances)[::-1][:15]

plt.subplot(2, 1, 2)
sns.barplot(x=xgb_importances[xgb_indices], y=[features[i] for i in xgb_indices], palette='plasma')
plt.title('XGBoost Feature Importance (Top 15)')
plt.xlabel('Relative Importance')

plt.tight_layout()
plt.show()

### Save Best Model for Streamlit Deployment

In [ ]:
best_model_path = f'../Models/{best_model_name}.pkl'
streamlit_model_path = '../Streamlit-app/fraud_model.pkl'

shutil.copy(best_model_path, streamlit_model_path)
print(f"Saved best model ({best_model_name}) to {streamlit_model_path} for deployment.")

### Discussion & Evaluation Report Summary

#### Why ROC-AUC and Recall matter more than Accuracy here:
In fraud detection, the cost of missing fraud (a false negative, leading to lower **Recall**) is significantly higher than the cost of flagging a legitimate transaction (a false positive). Furthermore, because the dataset is imbalanced (5% fraud), a naive model that predicts all transactions as legitimate would achieve an accuracy of 95% while catching zero fraud cases. Thus, F1-Score, Recall, and **ROC-AUC** are the core performance metrics.